<a href="https://colab.research.google.com/github/Saanvi-gug/Payload-SaanviGuglani/blob/main/Desert_Segmentation_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install the "Ready-to-use" AI models and Image Filter libraries
!pip install segmentation-models-pytorch albumentations

In [ ]:
# Create a folder for the dataset
!mkdir -p /content/dataset

# Unzip the file from your Drive to the local Colab storage
# Replace the path below with your actual zip file path (Right-click zip > Copy Path)
!unzip -q "/content/drive/MyDrive/Desert_project/Offroad_Segmentation_Training_Dataset.zip" -d "/content/dataset"

# List the folders to make sure it worked
!ls /content/dataset

In [ ]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np

class DesertDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.images = sorted(os.listdir(images_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.images[idx])
        mask_path = os.path.join(self.masks_dir, self.images[idx])

        image = np.array(Image.open(img_path).convert("RGB"))
        mask = np.array(Image.open(mask_path).convert("L"))

        # IMPORTANT: Convert large values (100, 200, 255) to simple IDs (0, 1, 2...)
        # This is a simple version; if you have 10 classes, we map them accordingly.
        unique_vals = [0, 100, 200, 255] # Add all values you saw in torch.unique
        for i, val in enumerate(unique_vals):
            mask[mask == val] = i

        if self.transform:
            augmentations = self.transform(image=image, mask=mask)
            image = augmentations["image"]
            mask = augmentations["mask"]

        return image, mask.long()

In [ ]:
import segmentation_models_pytorch as smp

# We use ResNet34 as the "encoder" (the brain) and U-Net as the "decoder" (the artist)
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",     # Starts with pre-learned knowledge
    in_channels=3,                  # RGB images
    classes=10,                     # Your 10 desert classes
)

# Move the model to the GPU you enabled
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
# Replace these with the actual paths you copied from the sidebar
TRAIN_IMG_DIR = '/content/dataset/Offroad_Segmentation_Training_Dataset/train/Color_Images' # <--- Verify this path!
TRAIN_MASK_DIR = '/content/dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation'

VAL_IMG_DIR = '/content/dataset/Offroad_Segmentation_Training_Dataset/val/Color_Images'
VAL_MASK_DIR = '/content/dataset/Offroad_Segmentation_Training_Dataset/val/Segmentation'

# Now create the datasets again
train_dataset = DesertDataset(images_dir=TRAIN_IMG_DIR,
                              masks_dir=TRAIN_MASK_DIR,
                              )

val_dataset = DesertDataset(images_dir=VAL_IMG_DIR,
                            masks_dir=VAL_MASK_DIR,
                            )

# Training transformations: Keep it varied!
train_transform = A.Compose([
    A.Resize(256, 256),              # Resize to a consistent square
    A.HorizontalFlip(p=0.5),         # Randomly flip
    A.RandomBrightnessContrast(p=0.2), # Random lighting changes
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), # Standard AI normalization
    ToTensorV2(),
])

# Validation transformations: Keep it simple (No random flips here)
val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [ ]:
!find /content/dataset -maxdepth 3 -type d

In [ ]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)

In [ ]:
# Based on your sidebar screenshot:
TRAIN_IMG_DIR = '/content/dataset/Offroad_Segmentation_Training_Dataset/train/Color_Images'
TRAIN_MASK_DIR = '/content/dataset/Offroad_Segmentation_Training_Dataset/train/Segmentation'

VAL_IMG_DIR = '/content/dataset/Offroad_Segmentation_Training_Dataset/val/Color_Images'
VAL_MASK_DIR = '/content/dataset/Offroad_Segmentation_Training_Dataset/val/Segmentation'

# Verify if paths are correct (should print a list of files)
import os
print("Train images found:", len(os.listdir(TRAIN_IMG_DIR)))

In [ ]:
from torch.utils.data import DataLoader

# Re-initialize the dataset objects with the fixed paths
train_dataset = DesertDataset(images_dir=TRAIN_IMG_DIR,
                              masks_dir=TRAIN_MASK_DIR,
                              transform=train_transform)

val_dataset = DesertDataset(images_dir=VAL_IMG_DIR,
                            masks_dir=VAL_MASK_DIR,
                            transform=val_transform)

# Define the loaders (This creates the 'train_loader' variable)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

print("Dataloaders successfully created!")

In [ ]:
import matplotlib.pyplot as plt

# Get one batch of data
images, masks = next(iter(train_loader))

# Show the first 3 images and masks
plt.figure(figsize=(12, 6))
for i in range(3):
    # Image (Needs to be "un-normalized" to look normal)
    plt.subplot(2, 3, i+1)
    img = images[i].permute(1, 2, 0).cpu().numpy()
    img = (img * 0.229) + 0.485  # Reverse normalization
    plt.imshow(img.clip(0, 1))
    plt.title("Desert Image")
    plt.axis('off')

    # Mask
    plt.subplot(2, 3, i+4)
    plt.imshow(masks[i].cpu().numpy(), cmap='nipy_spectral')
    plt.title("Segmentation Mask")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Run this to find every single color value used in your masks
all_values = set()
for _, mask in val_loader:
    unique_in_batch = torch.unique(mask).numpy()
    all_values.update(unique_in_batch)

print("Actual Class Values in Dataset:", sorted(list(all_values)))

In [ ]:
model.eval()
with torch.no_grad():
    images, masks = next(iter(val_loader))
    outputs = model(images.to(device))
    preds = torch.argmax(outputs, dim=1)

# Visualize the first image in the batch
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1); plt.imshow(images[0].permute(1,2,0).cpu().numpy()); plt.title("Original")
plt.subplot(1, 3, 2); plt.imshow(masks[0].cpu().numpy()); plt.title("Correct Answer")
plt.subplot(1, 3, 3); plt.imshow(preds[0].cpu().numpy()); plt.title("AI's Guess")
plt.show()

In [ ]:
# Check the unique values in a single mask
sample_mask = next(iter(val_loader))[1]
print("Unique values in mask:", torch.unique(sample_mask))

In [ ]:
# Final Step: Predicting on the Unseen Test Set
model.eval()
test_images = os.listdir('/content/dataset/Offroad_Segmentation_Training_Dataset/train/Color_Images')
# (Code to loop through and save predictions goes here)

In [ ]:
# Example weights for 10 classes (You should calculate these based on your data)
# Give rare classes like Flowers (ID 5) and Logs (ID 6) much higher numbers
class_weights = torch.tensor([2.0, 3.0, 1.5, 3.0, 4.0, 10.0, 8.0, 5.0, 0.5, 0.5]).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
model.eval()
with torch.no_grad():
    for images, masks in val_loader:
        # MOVE TO GPU HERE
        images, masks = images.to(device), masks.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        # Now the math will work because everything is on 'cuda:0'
        for cls in range(10):
            intersection = ((preds == cls) & (masks == cls)).sum().item()
            union = ((preds == cls) | (masks == cls)).sum().item()
            # ... rest of your code

In [ ]:
import numpy as np

def calculate_miou(model, loader, device, num_classes=10):
    model.eval()
    ious = []

    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            # Calculate IoU for each class in the batch
            for cls in range(num_classes):
                intersection = ((preds == cls) & (masks == cls)).sum().item()
                union = ((preds == cls) | (masks == cls)).sum().item()
                if union > 0:
                    ious.append(intersection / union)

    return np.mean(ious)

# Run evaluation on the validation set
miou_score = calculate_miou(model, val_loader, device)
print(f"Validation mIoU Score: {miou_score:.4f}")

In [ ]:
# --- 1. SETTINGS ---
num_epochs = 60          # Increased study time
learning_rate = 1e-4     # Slower, more careful learning to fix 'static noise'

# --- 2. OPTIMIZER & LOSS ---
# We redefine these to apply the new learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.CrossEntropyLoss()

# --- 3. THE LOOP ---
print(f"Starting training for {num_epochs} epochs...")

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for images, masks in train_loader:
        # Move everything to GPU to avoid the 'RuntimeError' you saw earlier
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Print progress so you can track the Loss dropping
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {train_loss/len(train_loader):.4f}")

print("Training Complete!")

In [ ]:
# The correct path to your Google Drive folder
SAVE_PATH = '/content/drive/MyDrive/Desert_project/best_model.pth'

# Use .state_dict() (no double 'state')
torch.save(model.state_dict(), SAVE_PATH)

print(f"Model saved successfully to: {SAVE_PATH}")

In [ ]:
import os
import torch
import torchvision.transforms as T
from PIL import Image
import numpy as np

# 1. Setup paths
TEST_IMG_DIR = '/content/dataset/Offroad_Segmentation_Training_Dataset/test/Color_Images'
SAVE_DIR = '/content/submission_masks'
os.makedirs(SAVE_DIR, exist_ok=True)

# 2. Load the model (Ensure you've run the 'model = smp.Unet...' cell first)
model.load_state_dict(torch.load('/content/drive/MyDrive/Desert_project/best_model.pth'))
model.to(device)
model.eval()

# 3. Define the same transformation used in validation (Resize + Normalize)
transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

print(f"Generating predictions for {len(os.listdir(TEST_IMG_DIR))} images...")

with torch.no_grad():
    for img_name in os.listdir(TEST_IMG_DIR):
        img_path = os.path.join(TEST_IMG_DIR, img_name)

        # Load and transform image
        raw_image = Image.open(img_path).convert("RGB")
        input_tensor = transform(raw_image).unsqueeze(0).to(device) # Add batch dimension

        # Predict
        output = model(input_tensor)
        prediction = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()

        # Convert prediction back to original size (if needed by hackathon rules)
        # Note: We save as a 1-channel PNG so judges can read the exact ID (0-9)
        res = Image.fromarray(prediction.astype(np.uint8))
        res = res.resize(raw_image.size, resample=Image.NEAREST)

        # Save with the EXACT same filename
        save_path = os.path.join(SAVE_DIR, img_name.replace(".jpg", ".png"))
        res.save(save_path)

# 4. Zip the results for submission
!zip -r submission_results.zip /content/submission_masks
print("Finished! Download 'submission_results.zip' from the files sidebar.")